## RUL 라벨링

- 진동데이터 마지막 수집 데이터 60개 미만 > 정확한 고장 시점
- 진동데이터 마지막 1분 수집데이터 60개 > 예측 고장 시점(마지막 고장 시점+1초)
- 운전데이터 마지막 1초 수집데이터 공정 조건 실패 > 정확한 고장 시점
- 운전데이터 마지막 1초 수집데이터 공정 조건 실패X > 예측 고장 시점(마지막 고장 시점 + 1초)
>> 진동데이터 마지막 고장 시점 vs 운전데이터 마지막 고장시점       
: 둘 중 정확한 고장 시점이 있는 경우 이를 우선(일치하는지 확인)       
: 둘 다 정확한 고장 시점이 없는 경우 마지막 고장 시점 +1초 중 빠른 시간을 고장 시점으로 저장        
-> 이를 통해 timetofailure 확정      

In [49]:
import pandas as pd 
import numpy as np
import glob 
from nptdms import TdmsFile, TdmsWriter, ChannelObject
import os
import re 

### 최종) TrainSet별 TTF / RUL 라벨링

In [52]:
# trainset number 가져오는 함수
def get_train_num(file_name) :
    return int(re.findall(r'\d+', file_name)[0])

In [ ]:
# TTF 구하기(actual vs expect)
# RUL 구하기
def label_trainset(base_path, fs=25600) :
    # 1. Operation 파일 탐색
    op_files = sorted(glob.glob(os.path.join(base_path, "Train*_Operation.csv")))

    for op_path in op_files :
        if "_labeled" in op_path :
            continue 
        t_num = get_train_num(os.path.basename(op_path))
        print(f'\n>>> Processing Train {t_num}')

        # TTF 확정 로직
        df_op = pd.read_csv(op_path, encoding= 'cp949')
        vib_dir = os.path.join(base_path, f'Train{t_num}_Vibration')
        all_vib_files = glob.glob(os.path.join(vib_dir, '*.tdms'))
        vib_files = sorted([f for f in all_vib_files if "_labeled" not in f],
                           key = lambda x : int(os.path.basename(x).split('.')[0]))
        # 1) 운전 데이터 기반 TTF
        cond_temp = (df_op['  TC SP Front[℃]'] >= 200) | (df_op['  TC SP Rear[℃]'] >= 200)
        cond_torque = (df_op['  Torque[Nm]'] <= -20)
        cond_fail = cond_temp | cond_torque 

        last_op_time = df_op['Time[sec]'].iloc[-1]
        actual_failure_time_op = None 
        expect_failure_time_op = None 

        if cond_fail.any() :
            idx_fail = cond_fail.idxmax()
            actual_failure_time_op = df_op.loc[idx_fail, 'Time[sec]']
        else :
            # 고장 조건 미충족 시 +1초로 예측 
            expect_failure_time_op = last_op_time + 1

        # 2) 진동 데이터 기반 TTF
        vib_len_1min = fs * 60 # 1분동안 수집되는 진동 데이터

        last_vib_path = vib_files[-1]
        last_vib_num = int(os.path.basename(last_vib_path).split('.')[0])

        with TdmsFile.read(last_vib_path) as tdms :
            data_len = len(tdms.groups()[0].channels()[0]) # 마지막 파일의 CH1 행길이

        actual_failure_time_vib = None 
        expect_failure_time_vib = None 

        if data_len < vib_len_1min :
            # 60초 미만 수집
            actual_failure_time_vib = (last_vib_num - 1) * 600 + (data_len / fs) 
        else :
            # 60초 수집
            expect_failure_time_vib = (last_vib_num - 1) * 600 + 60 + 1
        
        # 3) 최종 TTF 확정 
        # actual이 하나라도 있다면 actual을 선택
        # 둘 다 actual이 없다면 expect 중 '큰 값'을 선택(데이터가 존재하는 마지막 지점을 신뢰)
        if actual_failure_time_op or actual_failure_time_vib :
            if actual_failure_time_op and actual_failure_time_vib :
                if actual_failure_time_op == actual_failure_time_vib :
                    print("[Check] 두 데이터의 고장 시점이 일치합니다.") 
                else :
                    print("[Check] 두 데이터의 고장 시점이 불일치합니다.") 
            ttf = min(filter(None, [actual_failure_time_op, actual_failure_time_vib]))
        else :
            ttf = max(filter(None, [expect_failure_time_op, expect_failure_time_vib]))
            print("[Check] 예측 고장 시점을 사용합니다.")
        
        # 4) Operation csv 업데이트
        df_op['ttf'] = ttf 
        df_op['RUL'] = (ttf - df_op['Time[sec]']).clip(lower= 0)
        df_op.to_csv(op_path.replace('.csv', '_labeled.csv'), index= False)
        print(f">> Final TTF: {ttf} sec")

        # 5) TDMS 채널 업데이트
        for fpath in vib_files :
            f_num = int(os.path.basename(fpath).split('.')[0])
            with TdmsFile.read(fpath) as tdms :
                group = tdms.groups()[0]
                group_name = group.name 

                new_fpath = fpath.replace('.tdms', '_labeled.tdms')

                num_samples = len(group.channels()[0])
                start_time = (f_num - 1) * 600
                current_time = start_time + (np.arange(num_samples) / fs)

                vib_RUL = (ttf - current_time).clip(min=0)
                vib_ttf = np.full(num_samples, ttf) 

                # 새 TDMS
                with TdmsWriter(new_fpath) as tdms_writer :
                    new_channels= []

                    for channel in group.channels() :
                        ch_data = channel[:]
                        new_channels.append(ChannelObject(group_name, channel.name, ch_data))

                    # 새 채널(RUL)
                    new_channels.append(ChannelObject(group_name, 'ttf', vib_ttf))
                    new_channels.append(ChannelObject(group_name, 'RUL', vib_RUL))

                    tdms_writer.write_segment(new_channels)
            print(f"[Vib] {f_num}_labeled.tdms 저장 완료")


In [58]:
label_trainset(base_path= '../data/Train')


>>> Processing Train 1
>> Final TTF: 75251 sec

>>> Processing Train 2
>> Final TTF: 67979 sec

>>> Processing Train 3
>> Final TTF: 53225 sec

>>> Processing Train 4
>> Final TTF: 82613 sec


In [ ]:
# >> Final TTF: 75251 sec
# >> Final TTF: 67979 sec
# >> Final TTF: 53225 sec
# >> Final TTF: 82613 sec


In [50]:
# 경로 설정
op_path = '../data/Train/Train1_Operation.csv'
vib_dir = '../data/Train/Train1_Vibration/'

## Train1 TEST

### 1. 운전 데이터 기반 고장 시점 확인

In [15]:
df_op = pd.read_csv(op_path, encoding= 'cp949')
last_op_row = df_op.iloc[-1]
last_op_time = last_op_row['Time[sec]']

# print('last_op_row', last_op_row)
print('last_op_time', last_op_time) #75251

# 고장 조건 확인
torque_col = df_op.columns.tolist()[1]
front_temp_col = df_op.columns.tolist()[3]
rear_temp_col = df_op.columns.tolist()[4]

cond_fail = (last_op_row[front_temp_col] >= 200) | (last_op_row[rear_temp_col] >= 200) | (last_op_row[torque_col] <= -20)

actual_failure_time_op = last_op_time if cond_fail else None
expect_failure_time_op = (last_op_time + 1) if not cond_fail else None

print("운전 데이터 실제 고장 시점 : ", actual_failure_time_op)
print("운전 데이터 예측 고장 시점 : ", expect_failure_time_op)

last_op_time 75251.0
운전 데이터 실제 고장 시점 :  75251.0
운전 데이터 예측 고장 시점 :  None


### 2. 진동 데이터 기반 고장 시점 확인

In [45]:
vib_files = sorted(glob.glob(os.path.join(vib_dir, '*.tdms')))
last_file_path = vib_files[-1]
last_file_num = int(os.path.basename(last_file_path).split('.')[0])

last_file_num # 126

# group[0] = vibration
# channels[0] = ch1
with TdmsFile.read(last_file_path) as tdms : 
    # 첫 번째 그룹의 첫 번째 채널 데이터 개수 확인
    group = tdms.groups()[0]
    channel = group.channels()[0]
    data_len = len(channel) # 1536000 > 25.6kHz

# data_len # 153600 = 25600 * 60sec
full_minute_samples = 25600 * 60
last_vib_time = (last_file_num-1)*600+60
# last_vib_time # 75060 

if data_len < full_minute_samples :
    # 60초 미만 수집됨 : 실제 고장 시점 계산 (시작점 + 수집된 초만큼)
    actual_failure_time_vib = (last_file_num - 1) * 600 + (data_len / 25600)
    expect_failure_time_vib = None 
else :
    # 60초 수집됨 : 다음 수집 주기 중 가장 빠른 시간인 + 1초를 예측 시점으로 설정
    actual_failure_time_vib = None 
    expect_failure_time_vib = last_vib_time + 1

actual_failure_time_vib 
expect_failure_time_vib # 75061

75061

### 3. 최종 고장 시점 결정

In [48]:
# 우선순위 : actual이 있다면 그 값을 사용, 둘 다 없다면 expect 중 빠른 값
if actual_failure_time_op or actual_failure_time_vib :
    if actual_failure_time_op and actual_failure_time_vib:
        if actual_failure_time_op == actual_failure_time_vib :
            print("[Check] TRUE : 둘 다 실제 고장 시점이 존재하고 일치합니다.")
        else :
            diff = abs(actual_failure_time_op, actual_failure_time_vib)
            print(f"[Check] WARNING : 둘 다 실제 고장 시점이 존재하는데 일치하지 않습니다.")
    time_to_failure = min(filter(None, [actual_failure_time_op, actual_failure_time_vib]))
else :
    # actual이 둘 다 없는 경우 빠른 expect값 선택
    time_to_failure = min(filter(None, [expect_failure_time_op, expect_failure_time_vib]))

print(f"--- Train 1 고장 분석 결과 ---")
print(f"Actual Failure (OP/Vib): {actual_failure_time_op} / {actual_failure_time_vib}")
print(f"Expect Failure (OP/Vib): {expect_failure_time_op} / {expect_failure_time_vib}")
print(f">> Final Time to Failure: {time_to_failure}")

--- Train 1 고장 분석 결과 ---
Actual Failure (OP/Vib): 75251.0 / None
Expect Failure (OP/Vib): None / 75061
>> Final Time to Failure: 75251.0
